# 09c — Feedforward Network on MNIST

Corresponds to: Note 09 — Deep Feedforward Network Architectures

**Libraries introduced in this notebook:** `torchvision.datasets.MNIST` — provides the MNIST handwritten digit dataset with automatic download and standard transforms.

**What this notebook demonstrates:**
1. Loading and inspecting real MNIST data (60,000 training images, 28×28 grayscale, 10 classes)
2. Building the [784, 512, 256, 128, 10] architecture from Note 09 Part 6
3. Training with cross-entropy loss and Adam optimiser
4. Evaluating on the test set — accuracy and per-class breakdown
5. Inspecting misclassified examples

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device:      {device}')
print(f'PyTorch:     {torch.__version__}')
print(f'Torchvision: {torchvision.__version__}')

## Part 1: Load MNIST

MNIST contains 60,000 training and 10,000 test images of handwritten digits (0–9). Each image is 28×28 pixels, grayscale. We flatten each image to a 784-dimensional vector — the input size for the Note 09 architecture.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
    transforms.Lambda(lambda x: x.view(-1)),   # flatten 28x28 -> 784
])

train_set = torchvision.datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_set  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=256, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False, num_workers=0)

print(f'Training samples: {len(train_set)}')
print(f'Test samples:     {len(test_set)}')
print(f'Input size:       {train_set[0][0].shape}  (flattened 28x28)')
print(f'Classes:          {train_set.classes}')

## Part 2: Build the Note 09 Architecture

The architecture from Note 09 Part 6: [784, 512, 256, 128, 10] with ReLU hidden activations. Parameter count verified in notebook 09a.

In [ ]:
model = nn.Sequential(
    nn.Linear(784, 512), nn.ReLU(),
    nn.Linear(512, 256), nn.ReLU(),
    nn.Linear(256, 128), nn.ReLU(),
    nn.Linear(128,  10),
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Architecture: [784, 512, 256, 128, 10]')
print(f'Total parameters: {total_params:,}')

## Part 3: Train

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def train_epoch(model, loader):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct += (out.argmax(1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n

def eval_epoch(model, loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            loss = criterion(out, y)
            total_loss += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            n += len(y)
    return total_loss / n, correct / n

print(f'{"Epoch":<8} {"Train Loss":<14} {"Train Acc":<14} {"Test Loss":<14} {"Test Acc"}')
print('-' * 60)

for epoch in range(1, 11):
    tr_loss, tr_acc = train_epoch(model, train_loader)
    te_loss, te_acc = eval_epoch(model, test_loader)
    print(f'{epoch:<8} {tr_loss:<14.4f} {tr_acc:<14.4f} {te_loss:<14.4f} {te_acc:.4f}')

## Part 4: Per-Class Accuracy

In [ ]:
model.eval()
class_correct = [0] * 10
class_total   = [0] * 10

with torch.no_grad():
    for X, y in test_loader:
        X, y = X.to(device), y.to(device)
        preds = model(X).argmax(1)
        for c in range(10):
            mask = (y == c)
            class_correct[c] += (preds[mask] == c).sum().item()
            class_total[c]   += mask.sum().item()

print('Per-class accuracy on test set:')
print(f'{"Digit":<8} {"Correct":<10} {"Total":<10} {"Accuracy"}')
print('-' * 38)
for c in range(10):
    print(f'{c:<8} {class_correct[c]:<10} {class_total[c]:<10} {class_correct[c]/class_total[c]:.4f}')

## Part 5: Misclassified Examples

In [ ]:
model.eval()
mistakes = []

with torch.no_grad():
    for X, y in test_loader:
        X, y = X.to(device), y.to(device)
        preds = model(X).argmax(1)
        wrong = (preds != y).nonzero(as_tuple=True)[0]
        for idx in wrong:
            mistakes.append((y[idx].item(), preds[idx].item()))
        if len(mistakes) >= 10:
            break

print(f'Total misclassified: {10000 - sum(class_correct)}')
print()
print('First 10 mistakes (true -> predicted):')
for true, pred in mistakes[:10]:
    print(f'  True: {true}  Predicted: {pred}')

## Summary

What this notebook demonstrated:

1. **MNIST** — 60,000 training images, 28×28 grayscale, flattened to 784-dim vectors. Normalised with MNIST mean (0.1307) and std (0.3081).
2. **Note 09 architecture** — [784, 512, 256, 128, 10] with ReLU activations. 567,434 parameters trained on real digit images.
3. **Training** — cross-entropy loss, Adam lr=1e-3, 10 epochs. A plain feedforward network achieves ~98% test accuracy on MNIST.
4. **Per-class accuracy** — some digits (e.g. 4 vs 9, 3 vs 5) are harder due to visual similarity.
5. **Misclassifications** — the network's errors are on genuinely ambiguous digits, not random failures.